In [ ]:
import os
from openai import OpenAI

from ddi.data import build_human
from ddi.manifest import write_dataset
from ddi.vocab import build_vocab
from ddi.prompt import make_specs, make_sample_fn, prompt_fingerprint
from ddi.synth import generate_raw, build_dataset_from_raw
from ddi.run import run_training

In [ ]:
# one-time vocab build

train, dev, val = build_human()
dev_id = write_dataset(dev, provenance="human:dev", seed=42, notes="dev - iterate here")
val_id = write_dataset(val, provenance="human:val", seed=42, notes="val - finalists only")
print(dev_id, val_id)
# 20260723-032552-e46d2b 20260723-032553-e79bce

In [ ]:
dev_id = "20260723-032552-e46d2b"
val_id = "20260723-032553-e79bce"

In [ ]:
# trainer config, same as the one used for the human data runs, bar the dataset name
cfg = {
    "model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
    "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
    "neg_ratio": None, "seed": 0, "dataset": "synthetic",
}

SYNTH_VERSION = "v6"

GEN_CFG  = {"model": "gpt-oss-120b", "temperature": 0.7,
            "reasoning_effort": "low", "max_output_tokens": 3000}
SPEC_CFG = {"n": 2000, "seed": 0}          # add n_drugs_dist / label_dist later when sweeping


In [ ]:
client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=120.0)

vocab     = build_vocab()
specs     = make_specs(vocab=vocab, **SPEC_CFG)
sample_fn = make_sample_fn(client, **GEN_CFG)

generate_raw(specs, sample_fn, gen_id=SYNTH_VERSION, max_workers=16)
synth_id, stats = build_dataset_from_raw(
    SYNTH_VERSION,
    generator={**GEN_CFG, "specs": SPEC_CFG, "prompt_sha": prompt_fingerprint()},
    vocab_source=vocab.fingerprint(), seed=SPEC_CFG["seed"])
print(stats)

# inspect before training
from ddi.manifest import load_dataset
inst, man = load_dataset(synth_id)

for r in inst[:10]:
    print(f"{r['label']:10s} {r['source']:10s} {r['text']}")


In [ ]:
none_ratio = man["label_distribution"]["NONE"] / sum(man["label_distribution"].values())
print(f"NONE ratio: {none_ratio:.3f} ({man['label_distribution']})")

In [ ]:

# --- train (once the 50 look right, regenerate at ~2000 first) --------------
run_id, metrics = run_training(synth_id, dev_id, cfg, notes="synthetic-only v1")
print(metrics["micro_f1_pos"])